01. Limpiador PLAN LAM

In [1]:
# 1. Limpieza y consolidación de archivos Excel con pandas
import os
import pandas as pd
from glob import glob
from datetime import datetime
import re

# Carpeta de entrada y salida
input_folder = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\LATAM\Inputs"
output_folder = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\LATAM\Outputs"

# Hojas a leer
dias = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]

# Buscar todos los archivos Excel con extensión xlsm
excel_files = glob(os.path.join(input_folder, "*.xlsm"))

df_list = []
for file in excel_files:
    try:
        xl = pd.ExcelFile(file)
        for dia in dias:
            if dia in xl.sheet_names:
                # Leer la celda A1 para la fecha
                fecha_raw = xl.parse(dia, header=None, usecols="A").iloc[0,0] if not xl.parse(dia, header=None, usecols="A").empty else None
                # Formatear la fecha como dd/mm/aaaa
                try:
                    Fecha = pd.to_datetime(fecha_raw, dayfirst=True).strftime('%d/%m/%Y') if pd.notnull(fecha_raw) else None
                except Exception:
                    Fecha = None
                # Promueve la segunda fila como nombres de columna
                df = xl.parse(dia, usecols="B:N", header=1)
                df["Archivo"] = os.path.basename(file)
                df["Hoja"] = dia
                df["Fecha"] = Fecha
                df_list.append(df)
    except Exception as e:
        print(f"Error leyendo {file}: {e}")

# Concatenar todos los DataFrames
if df_list:
    df_final = pd.concat(df_list, ignore_index=True)
    # Eliminar filas que estén vacías en todas las columnas
    df_final = df_final.dropna(how='all')
    # Eliminar filas con NaN en 'Peso (Kg)' y resetear el índice
    if 'Peso (Kg)' in df_final.columns:
        df_final = df_final.dropna(subset=['Peso (Kg)'])
        df_final.reset_index(drop=True, inplace=True)
    # Eliminar columnas específicas
    columnas_a_quitar = ['Cliente', 'DESTINO', 'PARTIDAS', 'No. De Booking']
    df_final = df_final.drop(columns=[col for col in columnas_a_quitar if col in df_final.columns])
    # Procesar conjuntos por 'Ruta' y asignar 'Volumen (m3) real'
    if 'Ruta' in df_final.columns and 'Volumen (m3)' in df_final.columns:
        volumen_real = [None] * len(df_final)
        idx_inicio = df_final['Ruta'].notna()
        indices = df_final.index[idx_inicio].tolist() + [len(df_final)]
        for i in range(len(indices)-1):
            sub_idx = range(indices[i], indices[i+1])
            sub_df = df_final.iloc[sub_idx]
            ultimo_vol = sub_df['Volumen (m3)'].dropna().iloc[-1] if not sub_df['Volumen (m3)'].dropna().empty else None
            for j in sub_idx:
                volumen_real[j] = ultimo_vol
        df_final['Volumen (m3) real'] = volumen_real
    else:
        print("No se encontraron las columnas 'Ruta' y 'Volumen (m3)'.")
    # Asignar valores a la columna 'Tipo de pedido' según la columna 'Tipo'
    regular = ["NORMAL", "ATIPICO", "CDNA", "PROFORMA", "NORMAL/ ADICIONAL", "NORMAL/ADICIONAL", "ENVIO DE CAJAS"]
    refacciones = ["REFACCIONES", "REFACTURACION", "Catalogos traslado", "CATALOGOS"]
    def tipo_pedido(tipo):
        if pd.isna(tipo):
            return "Borrar"
        tipo_str = str(tipo).strip().upper()
        if tipo_str in [r.upper() for r in regular]:
            return "Regular"
        elif tipo_str in [r.upper() for r in refacciones]:
            return "Refacciones"
        else:
            return "Borrar"
    df_final['Tipo de pedido'] = df_final['Tipo'].apply(tipo_pedido)

    # Limpiar espacios específicos en 'Tipo de Unidad'
    if 'Tipo de Unidad' in df_final.columns:
        def limpiar_tipo_unidad(valor):
            if pd.isna(valor):
                return valor
            valor_str = str(valor)
            # Solo procesar valores que NO sean "Cliente recoge" o frases largas
            if len(valor_str.split()) <= 2:
                # Quitar espacios entre números y letras o letras y números
                # Ejemplos: "40 HC" -> "40HC", "C 53" -> "C53"
                valor_limpio = re.sub(r'(\w)\s+(\w)', r'\1\2', valor_str)
                return valor_limpio
            else:
                # Mantener frases largas como "Cliente recoge" sin cambios
                return valor_str
        
        df_final['Tipo de Unidad'] = df_final['Tipo de Unidad'].apply(limpiar_tipo_unidad)
        
        # Eliminar filas donde 'Tipo de Unidad' sea exactamente 'Clienterecoge'
        df_final = df_final[df_final['Tipo de Unidad'] != 'Clienterecoge']
        df_final.reset_index(drop=True, inplace=True)
    
    # Convertir la columna 'Fecha' de string a tipo datetime
    if 'Fecha' in df_final.columns:
        df_final['Fecha'] = pd.to_datetime(df_final['Fecha'], format='%d/%m/%Y', errors='coerce')
    
    # Ordenar columnas según el orden solicitado
    column_order = [
        "Codigo (RuEnTra)",
        "Ruta",
        "Pedido",
        "Tipo",
        "Entrega",
        "N° Transporte",
        "Peso (Kg)",
        "Volumen (m3)",
        "Tipo de Unidad",
        "Tipo de carga",
        "Volumen (m3) real",
        "Fecha",
        "Tipo de pedido",
        "Archivo",
        "Hoja"
    ]
    column_order_exist = [col for col in column_order if col in df_final.columns]
    df_final = df_final[column_order_exist]

    # Rellenar celdas vacías con el valor de la celda anterior inmediata (hacia arriba)
    df_final = df_final.fillna(method='ffill')

    # Reemplazar 'DIRECTA' en 'N° Transporte' por el valor anterior inmediato
    if 'N° Transporte' in df_final.columns:
        mask_directa = df_final['N° Transporte'].astype(str).str.upper() == 'DIRECTA'
        df_final.loc[mask_directa, 'N° Transporte'] = None
        df_final['N° Transporte'] = df_final['N° Transporte'].fillna(method='ffill')

    # Eliminar filas donde 'Tipo de pedido' sea 'Borrar'
    df_final = df_final[df_final['Tipo de pedido'] != 'Borrar']

    # Ordenar por Fecha del más viejo al más reciente
    if 'Fecha' in df_final.columns:
        df_final['Fecha_orden'] = pd.to_datetime(df_final['Fecha'], format='%d/%m/%Y', errors='coerce')
        df_final = df_final.sort_values('Fecha_orden', ascending=True).drop(columns=['Fecha_orden']).reset_index(drop=True)

    # Resumen por fecha: cuántas filas hay por fecha
    resumen_fecha = df_final.groupby('Fecha').size().reset_index(name='Cantidad de filas')

    # Resumen por archivo: qué días hay por archivo
    resumen_archivo = df_final.groupby('Archivo')['Hoja'].unique().reset_index()
    resumen_archivo['Días'] = resumen_archivo['Hoja'].apply(lambda x: ', '.join(x))
    resumen_archivo = resumen_archivo[['Archivo', 'Días']]

    # Nueva hoja: rutas únicas y su fecha correspondiente
    if 'Ruta' in df_final.columns and 'Fecha' in df_final.columns:
        rutas_fechas = df_final[['Ruta', 'Fecha']].drop_duplicates().reset_index(drop=True)
    else:
        rutas_fechas = pd.DataFrame()

    # Generar nombre de archivo único por ejecución
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = os.path.join(output_folder, f"PLAN_LAM_limpio_{timestamp}.xlsx")
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_final.to_excel(writer, sheet_name='Datos', index=False)
        resumen_fecha.to_excel(writer, sheet_name='Resumen por fecha', index=False)
        resumen_archivo.to_excel(writer, sheet_name='Resumen por archivo', index=False)
        rutas_fechas.to_excel(writer, sheet_name='Rutas y Fechas', index=False)
    print(f"Archivo exportado a: {output_path}")
else:
    print("No se encontraron datos en los archivos especificados.")

# Eliminar filas donde 'Tipo de Unidad' sea exactamente 'Clienterecoge'
df_final = df_final[df_final['Tipo de Unidad'] != 'Clienterecoge']
df_final.reset_index(drop=True, inplace=True)

C:\Users\igcamposg\AppData\Local\Temp\ipykernel_11192\1578214395.py:129: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final = df_final.fillna(method='ffill')
C:\Users\igcamposg\AppData\Local\Temp\ipykernel_11192\1578214395.py:135: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final['N° Transporte'] = df_final['N° Transporte'].fillna(method='ffill')


Archivo exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\LATAM\Outputs\PLAN_LAM_limpio_20260407_090453.xlsx
